# Data Cleaning And Modeling

In [155]:
import pandas as pd
import numpy as np
import re

pd.set_option("display.max_columns", None)

In [156]:
enrol = pd.read_csv("../data/processed/enrolment_full.csv")
demo = pd.read_csv("../data/processed/demographic_full.csv")
bio = pd.read_csv("../data/processed/biometric_full.csv")
print(enrol.shape, demo.shape, bio.shape)

(1006029, 7) (2071700, 6) (1861108, 6)


In [157]:
valid_states = set(
    enrol["state"]
    .dropna()
    .str.strip()
    .str.lower()
    .unique()
)


In [158]:
district_to_state_map = {
    # Bihar
    "darbhanga": "bihar",

    # Telangana
    "balanagar": "telangana",

    # Maharashtra
    "nagpur": "maharashtra",

    # Rajasthan
    "jaipur": "rajasthan",

    # Tamil Nadu
    "raja annamalai puram": "tamil nadu",

    # Andhra Pradesh
    "madanapalle": "andhra pradesh",

    # Karnataka
    "puttenahalli": "karnataka",

    # Uttarakhand (legacy)
    "uttaranchal": "uttarakhand",

    # West Bengal typo
    "west bengli": "west bengal"
}


In [159]:
def normalize_state(state):
    if pd.isna(state):
        return state

    state = state.strip().lower()
    state = re.sub(r"&", "and", state)
    state = re.sub(r"\s+", " ", state)

    canonical_map = {
        # Tamil Nadu
        "tamil nadu": "tamil nadu",
        "tamilnadu": "tamil nadu",
        
        # Chhattisgarh
        "chhatisgarh": "chhattisgarh",
        "chhattisgargh": "chhattisgarh",

        # Odisha
        "orissa": "odisha",
        "odisha": "odisha",

        # West Bengal
        "west bengal": "west bengal",
        "westbengal": "west bengal",
        "west bangal": "west bengal",

        # Jammu & Kashmir
        "jammu and kashmir": "jammu and kashmir",

        # Andaman & Nicobar Islands
        "andaman and nicobar islands": "andaman and nicobar islands",

        # Puducherry
        "pondicherry": "puducherry",
        "puducherry": "puducherry",

        # Dadra & Nagar Haveli + Daman & Diu
        "dadra and nagar haveli": "dadra and nagar haveli and daman and diu",
        "daman and diu": "dadra and nagar haveli and daman and diu",
        "the dadra and nagar haveli and daman and diu":
            "dadra and nagar haveli and daman and diu",
        "dadra and nagar haveli and daman and diu":
            "dadra and nagar haveli and daman and diu"
    }

    # Step 1: canonical normalization
    normalized = canonical_map.get(state, state)

    # Step 2: district → state fallback
    if normalized not in valid_states:
        normalized = district_to_state_map.get(normalized, normalized)

    return normalized


In [160]:
for df in [enrol, demo, bio]:
    df["state"] = df["state"].apply(normalize_state)


In [161]:
for df in [enrol, demo, bio]:
    df.drop(df[df["state"] == "100000"].index, inplace=True)

In [162]:
print("Enrolment states:", enrol["state"].nunique())
print("Demographic states:", demo["state"].nunique())
print("Biometric states:", bio["state"].nunique())


Enrolment states: 36
Demographic states: 36
Biometric states: 36


In [163]:
def clean_columns(df):
    df.columns = (
        df.columns.str.lower()
        .str.strip()
        .str.replace(" ", "_")
    )
    return df

enrol = clean_columns(enrol)
demo  = clean_columns(demo)
bio   = clean_columns(bio)

In [164]:
for df in [enrol, demo, bio]:
    df["pincode"] = df["pincode"].astype(str).str.zfill(6)

In [165]:
dim_pincode = pd.concat([
    enrol[["pincode"]],
    demo[["pincode"]],
    bio[["pincode"]]
]).drop_duplicates().reset_index(drop=True)

dim_pincode.head()

,pincode
0,793121
1,560043
2,208001
3,202133
4,560016


In [166]:
dim_pincode["pincode"].duplicated().sum()

np.int64(0)

In [167]:
fact_enrolment = enrol.merge(
    dim_pincode,
    on="pincode",
    how="inner"
)

fact_enrolment.head()


,date,state,district,pincode,age_0_5,age_5_17,age_18_greater
0,02-03-2025,meghalaya,East Khasi Hills,793121,11,61,37
1,09-03-2025,karnataka,Bengaluru Urban,560043,14,33,39
2,09-03-2025,uttar pradesh,Kanpur Nagar,208001,29,82,12
3,09-03-2025,uttar pradesh,Aligarh,202133,62,29,15
4,09-03-2025,karnataka,Bengaluru Urban,560016,14,16,21


In [168]:
fact_demographic = demo.merge(
    dim_pincode,
    on="pincode",
    how="inner"
)

fact_demographic.head()


,date,state,district,pincode,demo_age_5_17,demo_age_17_
0,01-03-2025,uttar pradesh,Gorakhpur,273213,49,529
1,01-03-2025,andhra pradesh,Chittoor,517132,22,375
2,01-03-2025,gujarat,Rajkot,360006,65,765
3,01-03-2025,andhra pradesh,Srikakulam,532484,24,314
4,01-03-2025,rajasthan,Udaipur,313801,45,785


In [169]:
fact_biometric = bio.merge(
    dim_pincode,
    on="pincode",
    how="inner"
)

fact_biometric.head()


,date,state,district,pincode,bio_age_5_17,bio_age_17_
0,01-03-2025,haryana,Mahendragarh,123029,280,577
1,01-03-2025,bihar,Madhepura,852121,144,369
2,01-03-2025,jammu and kashmir,Punch,185101,643,1091
3,01-03-2025,bihar,Bhojpur,802158,256,980
4,01-03-2025,tamil nadu,Madurai,625514,271,815


In [170]:
assert fact_enrolment["pincode"].isnull().sum() == 0
assert fact_demographic["pincode"].isnull().sum() == 0
assert fact_biometric["pincode"].isnull().sum() == 0

print("Referential integrity maintained using PINCODE")


Referential integrity maintained using PINCODE


In [171]:
dim_pincode.to_csv("../data/cleaned/dim_pincode.csv", index=False)

fact_enrolment.to_csv("../data/cleaned/fact_enrolment.csv", index=False)
fact_demographic.to_csv("../data/cleaned/fact_demographic.csv", index=False)
fact_biometric.to_csv("../data/cleaned/fact_biometric.csv", index=False)
